# Module 11 - 실습 1: 검증 피라미드 구현

## 학습 목표
- LLM 출력 검증의 6계층 피라미드 구조를 이해한다
- JSON 파싱(Layer 0)과 Pydantic 스키마 검증(Layer 1)을 구현할 수 있다
- Confidence 게이팅(Layer 2)으로 낮은 신뢰도 결과를 필터링할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/11-quality-assurance.md` (섹션 1.2, 2.1, 2.2)

## 개념 요약: 6계층 검증 피라미드

```
          ┌─────────────────────┐
          │  Layer 6: 정량 평가  │  ← Golden Set
        ┌─┴─────────────────────┴─┐
        │  Layer 5: 인간 검증      │  ← HITL
      ┌─┴─────────────────────────┴─┐
      │  Layer 4: 외부 검증          │  ← AST/빌드
    ┌─┴───────────────────────────────┴─┐
    │  Layer 3: 자기 검증               │  ← Self-Reflection
  ┌─┴─────────────────────────────────────┴─┐
  │  Layer 2: 의미적 게이팅                  │  ← Confidence
┌─┴───────────────────────────────────────────┴─┐
│  Layer 1: 스키마 검증                           │  ← Pydantic
└───────────────────────────────────────────────────┘
              Layer 0: JSON 파싱
```

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
from typing import Type
from enum import Enum
from dataclasses import dataclass
from pydantic import BaseModel, Field, field_validator, ValidationError
from common.fake_llm import FakeLLM

print("검증 피라미드 실습 준비 완료")

## 실습 1: Layer 0 - JSON 파싱

LLM은 JSON을 다양한 형태로 출력합니다. 3단계 폴백 전략으로 JSON을 파싱하는 함수를 구현하세요:
1. 전체 텍스트를 직접 JSON 파싱
2. ` ```json ... ``` ` 마크다운 블록에서 추출
3. 첫 번째 `{` ~ 마지막 `}` 사이 추출

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: json.loads(text)로 직접 파싱 시도 (성공하면 바로 반환)
# 힌트 2: re.search(r'```(?:json)?\s*\n?(.*?)\n?\s*```', text, re.DOTALL) 로 마크다운 블록 추출
# 힌트 3: text.find("{")와 text.rfind("}") 로 중괄호 범위 추출

def parse_json_from_llm(text: str) -> dict:
    """LLM 출력에서 JSON을 추출합니다.
    
    3단계 폴백 전략으로 파싱합니다.
    """
    # 1단계: 직접 파싱 시도
    # TODO: 구현
    
    # 2단계: 마크다운 JSON 블록 추출
    # TODO: 구현
    
    # 3단계: 중괄호 범위 추출
    # TODO: 구현
    
    raise ValueError(f"JSON 파싱 실패: {text[:200]}...")

In [ ]:
# 검증 셀
# 케이스 1: 순수 JSON
result1 = parse_json_from_llm('{"key": "value", "num": 42}')
assert result1 == {"key": "value", "num": 42}, f"순수 JSON 파싱 실패: {result1}"

# 케이스 2: 마크다운 블록
result2 = parse_json_from_llm('다음은 결과입니다:\n```json\n{"summary": "분석 완료", "confidence": 0.85}\n```')
assert result2["summary"] == "분석 완료", f"마크다운 JSON 파싱 실패: {result2}"

# 케이스 3: 텍스트 안에 JSON
result3 = parse_json_from_llm('분석 결과: {"status": "ok", "score": 0.9} 이상입니다.')
assert result3["status"] == "ok", f"텍스트 내 JSON 파싱 실패: {result3}"

print("✅ 실습 1 완료! parse_json_from_llm이 3가지 형태를 모두 처리합니다.")

## 실습 2: Layer 1 - Pydantic 스키마 검증

LLM 분석 결과를 검증하는 Pydantic 스키마를 정의하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: BaseModel을 상속하여 AnalysisResult 클래스 정의
# 힌트 2: confidence는 ge=0.0, le=1.0 제약을 가짐
# 힌트 3: @field_validator로 confidence를 소수점 2자리로 반올림

class AnalysisResult(BaseModel):
    """LLM 분석 결과 스키마."""
    summary: str = Field(min_length=1, description="분석 요약")
    confidence: float = Field(ge=0.0, le=1.0, description="분석 신뢰도")
    issues: list[str] = Field(default_factory=list, description="발견된 문제")
    recommendation: str = Field(min_length=1, description="권장 조치")
    
    # TODO: @field_validator로 confidence 소수점 2자리 반올림 추가

In [ ]:
# 검증 셀
valid_data = {
    "summary": "로그인 API에 인증 누락 취약점이 발견되었습니다.",
    "confidence": 0.856,  # 반올림 필요
    "issues": ["토큰 검증 미수행"],
    "recommendation": "JWT 검증 미들웨어를 추가하세요."
}
result = AnalysisResult.model_validate(valid_data)
assert result.confidence == 0.86, f"confidence가 소수점 2자리로 반올림되어야 합니다. 현재: {result.confidence}"
assert result.summary == valid_data["summary"]

# 필드 누락 시 기본값 적용 확인
partial_data = {"summary": "분석 완료", "confidence": 0.9, "recommendation": "조치 필요"}
result2 = AnalysisResult.model_validate(partial_data)
assert result2.issues == [], f"issues 기본값이 빈 리스트여야 합니다. 현재: {result2.issues}"

print(f"confidence 반올림: 0.856 → {result.confidence}")
print("✅ 실습 2 완료! AnalysisResult 스키마가 올바르게 검증합니다.")

## 실습 3: Layer 2 - Confidence 게이팅

confidence 값에 따라 ACCEPT/RETRY/REJECT를 결정하는 함수를 구현하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: GateAction Enum: ACCEPT, RETRY, REJECT
# 힌트 2: confidence >= threshold_accept → ACCEPT
# 힌트 3: confidence >= threshold_reject → RETRY
# 힌트 4: 그 외 → REJECT

class GateAction(str, Enum):
    ACCEPT = "accept"
    RETRY = "retry"
    REJECT = "reject"

@dataclass
class GateResult:
    action: GateAction
    confidence: float
    message: str

def confidence_gate(
    confidence: float,
    threshold_accept: float = 0.7,
    threshold_reject: float = 0.4,
) -> GateResult:
    """Confidence 값에 따라 ACCEPT / RETRY / REJECT를 결정합니다."""
    # TODO: 구현
    pass

In [ ]:
# 검증 셀
assert confidence_gate(0.85).action == GateAction.ACCEPT, "0.85는 ACCEPT여야 합니다"
assert confidence_gate(0.55).action == GateAction.RETRY, "0.55는 RETRY여야 합니다"
assert confidence_gate(0.25).action == GateAction.REJECT, "0.25는 REJECT여야 합니다"
assert confidence_gate(0.7).action == GateAction.ACCEPT, "임계값 0.7은 ACCEPT여야 합니다"
assert confidence_gate(0.4).action == GateAction.RETRY, "임계값 0.4는 RETRY여야 합니다"

print(f"0.85 → {confidence_gate(0.85).action.value}")
print(f"0.55 → {confidence_gate(0.55).action.value}")
print(f"0.25 → {confidence_gate(0.25).action.value}")
print("✅ 실습 3 완료! confidence_gate가 올바르게 분류합니다.")

## 실습 4: 전체 파이프라인 조합

Layer 0~2를 조합한 `parse_and_validate` 함수를 구현하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: parse_json_from_llm()으로 JSON 파싱 (Layer 0)
# 힌트 2: AnalysisResult.model_validate()로 스키마 검증 (Layer 1)
# 힌트 3: confidence_gate()로 신뢰도 확인 (Layer 2)

def parse_and_validate(text: str) -> dict:
    """LLM 출력 텍스트를 파싱하고 검증합니다.
    
    Returns:
        {"result": dict, "gate": GateResult} 또는
        {"error": str} (실패 시)
    """
    # TODO: 구현
    pass

In [ ]:
# FakeLLM으로 테스트
fake_llm = FakeLLM(responses={
    "분석": '{"summary": "인증 누락 취약점", "confidence": 0.85, "issues": ["JWT 미검증"], "recommendation": "미들웨어 추가"}',
    "낮은": '{"summary": "불확실한 분석", "confidence": 0.2, "issues": [], "recommendation": "재검토 필요"}',
})

# 높은 신뢰도 케이스
llm_output_high = fake_llm.invoke("코드를 분석해주세요").content
result_high = parse_and_validate(llm_output_high)
print(f"높은 신뢰도 결과: {result_high}")

# 낮은 신뢰도 케이스 (직접 JSON 전달)
low_confidence_json = '{"summary": "불확실", "confidence": 0.2, "issues": [], "recommendation": "재검토"}'
result_low = parse_and_validate(low_confidence_json)
print(f"낮은 신뢰도 결과: {result_low}")

In [ ]:
# 검증 셀
assert result_high is not None, "parse_and_validate가 None을 반환합니다"
# 높은 신뢰도 → ACCEPT
if "gate" in result_high:
    assert result_high["gate"].action in [GateAction.ACCEPT, GateAction.RETRY], \
        f"높은 신뢰도는 ACCEPT 또는 RETRY여야 합니다. 현재: {result_high['gate'].action}"

# 낮은 신뢰도 → REJECT
if "gate" in result_low:
    assert result_low["gate"].action == GateAction.REJECT, \
        f"낮은 신뢰도(0.2)는 REJECT여야 합니다. 현재: {result_low['gate'].action}"

print("✅ 실습 4 완료! 전체 검증 파이프라인이 올바르게 작동합니다.")

## 정리

이번 실습에서 배운 내용:
1. **Layer 0**: JSON 파싱 - 3단계 폴백으로 다양한 형태의 LLM 출력을 처리
2. **Layer 1**: Pydantic 스키마 - 필수 필드와 타입을 검증
3. **Layer 2**: Confidence 게이팅 - 신뢰도 임계값으로 결과를 필터링

## 다음 모듈
- **실습 2**: `02_self_reflection.ipynb` - LangGraph Self-Reflection 루프